# 03 — ERCOT Weather-Enhanced Modeling and Risk Evaluation

This notebook trains and evaluates the final ERCOT hourly demand forecasting workflow using calendar, demand-history, and statewide weather features prepared in Notebook 02. Ridge, Random Forest, and HistGradientBoosting models are tuned with time-series cross-validation and evaluated on a 2025 holdout period.

Model selection is based primarily on overall MAE, with RMSE used to monitor larger forecast misses. Performance is then evaluated across normal, high, and extreme demand periods and across normal, hot, and cold temperature conditions.

# Load and Prepare Modeling Data

In [1]:
import duckdb
import pandas as pd

con=duckdb.connect('ercot_weather.duckdb')

df_model = con.sql('''
    SELECT *
    FROM demand_weather_model
    ORDER BY period_utc
''').df()

con.close()

df_model.head()

,period_utc,value,hour,dayofweek,month,year,lag_1,lag_24,lag_168,rolling_mean_24,...,extreme_demand_flag,temp_mean,temp_max,temp_min,humid_mean,dewpoint_mean,cloud_cover_mean,apparent_temp_mean,wind_speed_mean,precip_mean
0,2019-01-01 00:00:00-06:00,37301.0,6,1,1,2019,NaN,NaN,NaN,NaN,...,False,44.088047,54.128300,37.580002,83.323517,39.213051,20.166666,39.003639,6.370968,0.0
1,2019-01-01 01:00:00-06:00,36953.0,7,1,1,2019,37301.0,NaN,NaN,NaN,...,False,42.678051,55.478302,34.340000,87.261353,39.078049,31.833334,37.659748,6.213417,0.0
2,2019-01-01 02:00:00-06:00,37114.0,8,1,1,2019,36953.0,NaN,NaN,NaN,...,False,41.748051,56.918301,31.280001,90.389404,39.093052,28.500000,36.578342,6.587480,0.0
3,2019-01-01 03:00:00-06:00,37154.0,9,1,1,2019,37114.0,NaN,NaN,NaN,...,False,40.833050,50.888298,28.940001,88.685234,37.683048,27.833334,34.419422,8.797850,0.0
4,2019-01-01 04:00:00-06:00,37290.0,10,1,1,2019,37154.0,NaN,NaN,NaN,...,False,40.158051,52.868301,27.410000,86.765106,36.423050,11.500000,33.537193,8.912553,0.0


In [2]:
df_model.shape

(61345, 22)

In [3]:
df_model["period_utc"] = df_model["period_utc"].dt.tz_convert("UTC")

period_local = df_model["period_utc"].dt.tz_convert("America/Chicago")

df_model["local_hour"] = period_local.dt.hour
df_model["local_dayofweek"] = period_local.dt.dayofweek
df_model["local_month"] = period_local.dt.month
df_model["local_year"] = period_local.dt.year

In [4]:
df_model['is_weekend'] = df_model['dayofweek'].isin([5,6])
df_model['is_summer'] = df_model['month'].isin([6,7,8,9])
df_model['is_winter'] = df_model['month'].isin([12,1,2])

In [5]:
df_model.dtypes

period_utc             datetime64[us, UTC]
value                              float64
hour                                 int64
dayofweek                            int64
month                                int64
year                                 int64
lag_1                              float64
lag_24                             float64
lag_168                            float64
rolling_mean_24                    float64
rolling_mean_168                   float64
high_demand_flag                      bool
extreme_demand_flag                   bool
temp_mean                          float32
temp_max                           float32
temp_min                           float32
humid_mean                         float32
dewpoint_mean                      float32
cloud_cover_mean                   float32
apparent_temp_mean                 float32
wind_speed_mean                    float32
precip_mean                        float32
local_hour                           int32
local_dayof

Calendar features are recreated in Central Time so they align with ERCOT operating patterns. Weekend, summer, and winter flags provide broader calendar categories that may help the models distinguish weekly and seasonal demand regimes.

In [6]:
df_model.head()

,period_utc,value,hour,dayofweek,month,year,lag_1,lag_24,lag_168,rolling_mean_24,...,apparent_temp_mean,wind_speed_mean,precip_mean,local_hour,local_dayofweek,local_month,local_year,is_weekend,is_summer,is_winter
0,2019-01-01 06:00:00+00:00,37301.0,6,1,1,2019,NaN,NaN,NaN,NaN,...,39.003639,6.370968,0.0,0,1,1,2019,False,False,True
1,2019-01-01 07:00:00+00:00,36953.0,7,1,1,2019,37301.0,NaN,NaN,NaN,...,37.659748,6.213417,0.0,1,1,1,2019,False,False,True
2,2019-01-01 08:00:00+00:00,37114.0,8,1,1,2019,36953.0,NaN,NaN,NaN,...,36.578342,6.587480,0.0,2,1,1,2019,False,False,True
3,2019-01-01 09:00:00+00:00,37154.0,9,1,1,2019,37114.0,NaN,NaN,NaN,...,34.419422,8.797850,0.0,3,1,1,2019,False,False,True
4,2019-01-01 10:00:00+00:00,37290.0,10,1,1,2019,37154.0,NaN,NaN,NaN,...,33.537193,8.912553,0.0,4,1,1,2019,False,False,True


In [7]:
df_model.groupby('is_weekend')['dayofweek'].unique()

is_weekend
False    [1, 2, 3, 4, 0]
True              [5, 6]
Name: dayofweek, dtype: object

In [8]:
df_model.groupby('is_summer')['month'].unique()

is_summer
False    [1, 2, 3, 4, 5, 10, 11, 12]
True                    [6, 7, 8, 9]
Name: month, dtype: object

In [9]:
df_model.groupby('is_winter')['month'].unique()

is_winter
False    [3, 4, 5, 6, 7, 8, 9, 10, 11]
True                        [1, 2, 12]
Name: month, dtype: object

The engineered calendar flags contain the expected local days and months. Their predictive value will be determined through model evaluation rather than assumed from feature creation.

## Handle Initial Lag and Rolling Nulls

In [10]:
df_model.isna().sum()

period_utc               0
value                    0
hour                     0
dayofweek                0
month                    0
year                     0
lag_1                    1
lag_24                  24
lag_168                168
rolling_mean_24         24
rolling_mean_168       168
high_demand_flag         0
extreme_demand_flag      0
temp_mean                0
temp_max                 0
temp_min                 0
humid_mean               0
dewpoint_mean            0
cloud_cover_mean         0
apparent_temp_mean       0
wind_speed_mean          0
precip_mean              0
local_hour               0
local_dayofweek          0
local_month              0
local_year               0
is_weekend               0
is_summer                0
is_winter                0
dtype: int64

In [11]:
df_model_clean = df_model.dropna(
    subset = [
        'lag_1',
        'lag_24',
        'lag_168',
        'rolling_mean_24',
        'rolling_mean_168'
    ]
).copy()

df_model_clean.head()

,period_utc,value,hour,dayofweek,month,year,lag_1,lag_24,lag_168,rolling_mean_24,...,apparent_temp_mean,wind_speed_mean,precip_mean,local_hour,local_dayofweek,local_month,local_year,is_weekend,is_summer,is_winter
168,2019-01-08 06:00:00+00:00,32224.0,6,1,1,2019,34538.0,31505.0,37301.0,34802.333333,...,57.522842,7.279257,0.0,0,1,1,2019,False,False,True
169,2019-01-08 07:00:00+00:00,30574.0,7,1,1,2019,32224.0,30029.0,36953.0,34832.291667,...,56.926517,6.678637,0.0,1,1,1,2019,False,False,True
170,2019-01-08 08:00:00+00:00,29701.0,8,1,1,2019,30574.0,28963.0,37114.0,34855.000000,...,56.392902,5.894207,0.0,2,1,1,2019,False,False,True
171,2019-01-08 09:00:00+00:00,29397.0,9,1,1,2019,29701.0,28581.0,37154.0,34885.750000,...,54.640015,5.928419,0.0,3,1,1,2019,False,False,True
172,2019-01-08 10:00:00+00:00,29496.0,10,1,1,2019,29397.0,28518.0,37290.0,34919.750000,...,54.643185,5.114326,0.0,4,1,1,2019,False,False,True


In [12]:
df_model_clean.shape

(61177, 29)

The missing values are limited to the beginning of the time series, where the lag and rolling features do not yet have a demand history. The first 168 hours are removed because the weekly features require seven days of prior observations.

This removes approximately 0.27% of the available records. Imputation is not used as filling them would create artificial demand history. The remaining modeling table contains complete feature and target values.

## Define the Modeling Pipeline and Time Split

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

num_features = [
    'local_year',
    'lag_1',
    'lag_24',
    'lag_168',
    'rolling_mean_24',
    'rolling_mean_168',
    'temp_mean',
    'temp_max',
    'temp_min',
    'humid_mean',
    'dewpoint_mean',
    'cloud_cover_mean',
    'apparent_temp_mean',
    'wind_speed_mean',
    'precip_mean',
]

cat_features = [
    "local_hour",
    "local_dayofweek",
    "local_month"
]

bin_features = [
    'is_weekend',
    'is_summer',
    'is_winter'
]

ridge_preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        num_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        cat_features
    ),
    (
        "binary",
        "passthrough",
        bin_features
    )
])

def make_ridge_pipeline(model):
    return Pipeline([
        ('preprocessor', ridge_preprocessor),
        ('model', model)
    ])

def make_tree_pipeline(model):
        return Pipeline(steps =[
            ('model', model)
    ])

target = 'value'

feature_cols = num_features + cat_features + bin_features

X=df_model_clean[feature_cols]
y=df_model_clean[target]

train_mask = df_model_clean['local_year'] < 2025
test_mask =df_model_clean['local_year'] == 2025

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

Continuous demand-history and weather features are standardized for the Ridge model. Local hour, day of week, and month are one-hot encoded so the linear model does not assume that their numeric values have a straight-line relationship with demand. The same preprocessing workflow is applied consistently across all three models.

The target `value`, timestamp, and demand-risk fields are excluded from the model inputs.

In [14]:
splits = pd.DataFrame({
    'splits': ['train', 'test'],
    'rows': [train_mask.sum(), test_mask.sum()],
    'start': [
        df_model_clean.loc[X_train.index, 'period_utc'].min(),
        df_model_clean.loc[X_test.index, 'period_utc'].min()
    ],
    'end': [
        df_model_clean.loc[X_train.index, 'period_utc'].max(),
        df_model_clean.loc[X_test.index, 'period_utc'].max()
    ]
})

splits

,splits,rows,start,end
0,train,52440,2019-01-08 06:00:00+00:00,2025-01-01 05:00:00+00:00
1,test,8737,2025-01-01 06:00:00+00:00,2025-12-31 06:00:00+00:00


Data before 2025 is used for training and model selection, while 2025 is held out for final testing. No random split is used because that would allow later observations to inform predictions for earlier periods.

## Train and Tune Forecasting Models

Model hyperparameters are selected using five-fold time-series cross-validation within the training period. Each fold trains on earlier observations and validates on later observations, preserving the order required for forecasting. The 2025 holdout remains untouched until final model evaluation.

In [15]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

base_grid = {
    'model__alpha': [.01,.1,1,10,100]
}

base_search = GridSearchCV(
    estimator= make_ridge_pipeline(Ridge()),
    param_grid = base_grid,
    scoring = 'neg_mean_absolute_error',
    cv=tscv,
    n_jobs=-1
)
base_search.fit(X_train, y_train)
base_params, base_best_score = base_search.best_params_, -(base_search.best_score_)
best_base = base_search.best_estimator_
base_pred_train = best_base.predict(X_train)
base_pred_test = best_base.predict(X_test)

base_mae_train = mean_absolute_error(y_train, base_pred_train)
base_mae_test = mean_absolute_error(y_test, base_pred_test)
base_rmse_test = root_mean_squared_error(y_test, base_pred_test)
base_rmse_train = root_mean_squared_error(y_train, base_pred_train)
base_gap_mae = base_mae_test - base_mae_train
base_gap_rmse = base_rmse_test - base_rmse_train
print('Best model parameters are', base_params, f'Baseline MAE is: {base_mae_test:.2f}. Baseline RMSE is: {base_rmse_test:.2f}.')

Best model parameters are {'model__alpha': 1} Baseline MAE is: 857.10. Baseline RMSE is: 1122.20.


In [16]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

rf_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 5, 10],
    'model__max_features': ['sqrt', .5, 1],

}
rf_search = GridSearchCV(
    estimator = make_tree_pipeline(RandomForestRegressor(random_state=42)),
    param_grid = rf_grid,
    scoring = 'neg_mean_absolute_error',
    cv = tscv,
    n_jobs = -1
)

rf_search.fit(X_train, y_train)
rf_params, rf_best_score = rf_search.best_params_, -(rf_search.best_score_)
best_rf = rf_search.best_estimator_
rf_pred_train = best_rf.predict(X_train)
rf_pred_test = best_rf.predict(X_test)

rf_mae_train = mean_absolute_error(y_train, rf_pred_train)
rf_mae_test = mean_absolute_error(y_test, rf_pred_test)
rf_rmse_train = root_mean_squared_error(y_train, rf_pred_train)
rf_rmse_test = root_mean_squared_error(y_test, rf_pred_test)
rf_gap_mae = rf_mae_test - rf_mae_train
rf_gap_rmse = rf_rmse_test - rf_rmse_train
print('Best RF model parameters are', rf_params, f'RF MAE is: {rf_mae_test:.2f}. RF RMSE is: {rf_rmse_test:.2f}.')

Best RF model parameters are {'model__max_depth': 15, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 100} RF MAE is: 680.68. RF RMSE is: 933.92.


In [17]:
hgb_grid ={
    'model__max_iter': [100, 300, 500],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__max_leaf_nodes': [15, 31, 63],
    'model__min_samples_leaf': [20, 50, 100],
    'model__l2_regularization': [0.0, 0.1, 1.0]
}

hgb_search = GridSearchCV(
    estimator= make_tree_pipeline(HistGradientBoostingRegressor(random_state=42)),
    param_grid = hgb_grid,
    scoring = 'neg_mean_absolute_error',
    cv = tscv,
    n_jobs = -1
)

hgb_search.fit(X_train, y_train)
hgb_params, hgb_best_score = hgb_search.best_params_, hgb_search.best_score_
best_hgb = hgb_search.best_estimator_

hgb_pred_train = best_hgb.predict(X_train)
hgb_pred_test = best_hgb.predict(X_test)

hgb_mae_train = mean_absolute_error(y_train, hgb_pred_train)
hgb_mae_test = mean_absolute_error(y_test, hgb_pred_test)
hgb_rmse_train = root_mean_squared_error(y_train, hgb_pred_train)
hgb_rmse_test = root_mean_squared_error(y_test, hgb_pred_test)
hgb_gap_mae = hgb_mae_test - hgb_mae_train
hgb_gap_rmse = hgb_rmse_test - hgb_rmse_train
print('Best HGB model parameters are', hgb_params, f'HGB MAE is: {hgb_mae_test:.2f}. HGB RMSE is: {hgb_rmse_test:.2f}.')

Best HGB model parameters are {'model__l2_regularization': 1.0, 'model__learning_rate': 0.05, 'model__max_iter': 500, 'model__max_leaf_nodes': 63, 'model__min_samples_leaf': 20} HGB MAE is: 487.32. HGB RMSE is: 648.24.


## Evaluate Model Performance

The tuned models are evaluated on the untouched 2025 holdout period. MAE is the primary comparison metric because it represents the average forecast error in megawatts, while RMSE places greater weight on larger forecast misses.

In [18]:
models = ['Ridge', 'RF', 'HGB']
mae_train = [base_mae_train, rf_mae_train, hgb_mae_train]
mae_test = [base_mae_test, rf_mae_test, hgb_mae_test]
rmse_train = [base_rmse_train, rf_rmse_train, hgb_rmse_train]
rmse_test = [base_rmse_test, rf_rmse_test, hgb_rmse_test]
mae_gap = [base_gap_mae, rf_gap_mae, hgb_gap_mae]
rmse_gap = [base_gap_rmse, rf_gap_rmse, hgb_gap_rmse]

def final_results_table(models, mae_train, mae_test, mae_gap, rmse_train, rmse_test, rmse_gap):
    rows = []
    for model, train_mae, test_mae, gap_mae, train_rmse, test_rmse,  gap_rmse in zip(
            models, mae_train, mae_test, mae_gap, rmse_train, rmse_test,  rmse_gap):
        rows.append({
             'model':model,
            'mae_train': train_mae,
            'mae_test': test_mae,
            'gap_mae': gap_mae,
            'rmse_train': train_rmse,
            'rmse_test': test_rmse,
            'gap_rmse': gap_rmse
        })
    return pd.DataFrame(rows)

final_results_table(models, mae_train, mae_test, mae_gap, rmse_train, rmse_test, rmse_gap)

,model,mae_train,mae_test,gap_mae,rmse_train,rmse_test,gap_rmse
0,Ridge,827.619210,857.101523,29.482313,1077.313573,1122.202116,44.888543
1,RF,264.047956,680.676802,416.628846,362.070471,933.916166,571.845695
2,HGB,233.429959,487.321788,253.891829,311.221732,648.240187,337.018456


In [19]:
percentage_table = pd.DataFrame({
    'model': models,
    'mae_percentage': pd.Series(mae_test) / y_test.mean(),
    'rmse_percentage': pd.Series(rmse_test) / y_test.mean()
}
)

percentage_table.style.format({
    "mae_percentage": "{:.2%}",
    "rmse_percentage": "{:.2%}"
})

,model,mae_percentage,rmse_percentage
0,Ridge,1.54%,2.01%
1,RF,1.22%,1.68%
2,HGB,0.87%,1.16%


HGB produced the lowest overall MAE and RMSE, making it the strongest general-purpose forecasting model. Random Forest provided the next-best performance, while Ridge had higher overall error but remained a useful linear benchmark.

The overall metrics summarize performance across the full test period, but they may obscure accuracy in higher risk conditions which are more important operationally. The following analysis evaluates each model separately across normal, high, and extreme demand conditions.

## Performance by Demand Segment

Overall metrics can hide changes in performance during peak-demand conditions. The 2025 holdout is divided using P95 and P99 thresholds calculated from the training period, preventing the test data from influencing the segment definitions.

In [20]:
model_results = pd.DataFrame({
    'period_utc': df_model_clean.loc[X_test.index]['period_utc'].values,
    'y_true': y_test.values,
    'ridge_pred': base_pred_test,
    'rf_pred': rf_pred_test,
    'hgb_pred': hgb_pred_test

})
model_results

,period_utc,y_true,ridge_pred,rf_pred,hgb_pred
0,2025-01-01 06:00:00,43875.0,42478.492173,43273.277114,43173.628645
1,2025-01-01 07:00:00,43161.0,41822.040152,42411.396275,42257.734372
2,2025-01-01 08:00:00,43085.0,41630.366596,42233.626367,42687.647568
3,2025-01-01 09:00:00,42870.0,41926.084290,42357.722161,43140.926398
4,2025-01-01 10:00:00,42709.0,42151.393537,42620.641491,43432.725411
...,...,...,...,...,...
8732,2025-12-31 02:00:00,55192.0,54467.260955,53598.987893,54958.742626
8733,2025-12-31 03:00:00,55664.0,54869.354375,53886.406555,55162.857501
8734,2025-12-31 04:00:00,55464.0,54746.367723,54395.213010,54870.530776
8735,2025-12-31 05:00:00,54362.0,53406.570886,53215.237824,53528.520487


In [24]:
hi_demand_threshold = y_train.quantile(.95)
ex_demand_threshold = y_train.quantile(.99)

masks = {
    'normal': model_results['y_true'] < hi_demand_threshold,
    'high_demand': (
        (model_results['y_true'] >= hi_demand_threshold) &
    (model_results["y_true"] < ex_demand_threshold)),
    'extreme_demand': model_results['y_true'] >= ex_demand_threshold
}
masks.items()

dict_items([('normal', 0       True
1       True
2       True
3       True
4       True
        ... 
8732    True
8733    True
8734    True
8735    True
8736    True
Name: y_true, Length: 8737, dtype: bool), ('high_demand', 0       False
1       False
2       False
3       False
4       False
        ...  
8732    False
8733    False
8734    False
8735    False
8736    False
Name: y_true, Length: 8737, dtype: bool), ('extreme_demand', 0       False
1       False
2       False
3       False
4       False
        ...  
8732    False
8733    False
8734    False
8735    False
8736    False
Name: y_true, Length: 8737, dtype: bool)])

In [34]:
pred_cols = ['ridge_pred', 'rf_pred', 'hgb_pred']
def eval_segment(df, cols, masks):
    rows = []
    for segment_name, mask in masks.items():
        for col in cols:
            mae = mean_absolute_error(
                    df.loc[mask, 'y_true'],
                    df.loc[mask, col]
            )
            rmse = root_mean_squared_error(
                df.loc[mask, 'y_true'],
                df.loc[mask, col]
            )
            mae_percentage = mae / df.loc[mask,'y_true'].mean()
            rmse_percentage = rmse / df.loc[mask, 'y_true'].mean()
            mean_error = (df.loc[mask, col] - df.loc[mask, "y_true"]).mean()
            rows.append({
                'segment': segment_name,
                'model': col,
                'mae':  mae,
                'mae_percentage': mae_percentage,
                'rmse': rmse,
                'rmse_percentage': rmse_percentage,
                'mean_error': mean_error,
                "hours": mask.sum()
            })
    return rows

eval_results_demand = pd.DataFrame(eval_segment(model_results, pred_cols, masks)).sort_values(['segment','mae'], ascending=[True, True])
eval_results_demand.style.format({
    'mae': '{:,.2f}',
    'mae_percentage': '{:.2%}',
    'rmse': '{:,.2f}',
    'rmse_percentage': '{:.2%}'
    })

,segment,model,mae,mae_percentage,rmse,rmse_percentage,mean_error,hours
6,extreme_demand,ridge_pred,612.03,0.76%,824.22,1.02%,-305.010155,183
8,extreme_demand,hgb_pred,820.29,1.02%,"1,041.87",1.29%,-488.951089,183
7,extreme_demand,rf_pred,840.00,1.04%,"1,064.42",1.32%,-623.350438,183
5,high_demand,hgb_pred,682.37,0.92%,843.74,1.14%,-438.588856,970
4,high_demand,rf_pred,981.59,1.33%,"1,263.34",1.71%,-790.516722,970
3,high_demand,ridge_pred,"1,012.95",1.37%,"1,257.98",1.70%,-242.131200,970
2,normal,hgb_pred,454.34,0.86%,605.69,1.15%,-264.042106,7584
1,normal,rf_pred,638.34,1.21%,879.39,1.67%,-394.767397,7584
0,normal,ridge_pred,843.08,1.60%,"1,109.95",2.10%,37.139591,7584


HGB produced the lowest error during normal and high-demand periods, making it the strongest model across most operating conditions. Ridge performed best during P99+ extreme-demand periods, reducing extreme-demand MAE by approximately 25% compared with HGB.

All three models tended to underpredict, as shown by their negative mean errors. Ridge had the smallest underprediction bias in this segment, indicating that it generalized more effectively at the highest demand levels. HGB remains the strongest general-purpose model but ridge’s performance at the extreme tail is nevertheless operationally important because forecast underestimation during peak conditions may create greater planning and reliability risk.

## Performance by Temperature Condition

Model performance is also evaluated across temperature conditions because the EDA showed that demand risk increases during unusually hot and cold weather. Temperature bands are defined using the training-period distribution so the 2025 holdout does not influence their boundaries.

The temperature segments are:

- Cold: at or below the training-period P5 temperature
- Normal: between the training-period P5 and P95 temperatures
- Hot: at or above the training-period P95 temperature

In [31]:
model_results['temp_mean'] = df_model_clean.loc[y_test.index, 'temp_mean'].values
model_results.head()

,period_utc,y_true,ridge_pred,rf_pred,hgb_pred,temp_mean
0,2025-01-01 06:00:00,43875.0,42478.492173,43273.277114,43173.628645,49.383053
1,2025-01-01 07:00:00,43161.0,41822.040152,42411.396275,42257.734372,48.288052
2,2025-01-01 08:00:00,43085.0,41630.366596,42233.626367,42687.647568,47.268051
3,2025-01-01 09:00:00,42870.0,41926.084290,42357.722161,43140.926398,46.383053
4,2025-01-01 10:00:00,42709.0,42151.393537,42620.641491,43432.725411,45.738052


In [54]:
train_temps = df_model_clean.loc[y_train.index, 'temp_mean']

cold_threshold = train_temps.quantile(.05)
hot_threshold = train_temps.quantile(.95)

temp_masks = {
    'cold': model_results['temp_mean'] <= cold_threshold,
    'normal': (
        (model_results['temp_mean'] > cold_threshold) &
        (model_results['temp_mean'] < hot_threshold)
    ),
    'hot': model_results['temp_mean'] >= hot_threshold
}
temp_masks.keys()

dict_keys(['cold', 'normal', 'hot'])

In [55]:
eval_results_temp = pd.DataFrame(
    eval_segment(
        model_results, pred_cols, temp_masks
        )).sort_values(
            ['segment', 'mae'],
            ascending=[True, True]
        )

eval_results_temp.style.format({
    'mae': '{:,.2f}',
    'mae_percentage': '{:.2%}',
    'rmse': '{:,.2f}',
    'rmse_percentage': '{:.2%}'
    })

,segment,model,mae,mae_percentage,rmse,rmse_percentage,mean_error,hours
2,cold,hgb_pred,769.19,1.26%,972.53,1.60%,-438.244952,553
1,cold,rf_pred,"1,319.96",2.17%,"1,745.82",2.86%,-1024.349936,553
0,cold,ridge_pred,"1,537.62",2.52%,"1,810.86",2.97%,-680.314026,553
8,hot,hgb_pred,653.03,0.84%,832.60,1.07%,-220.697326,333
7,hot,rf_pred,711.01,0.92%,893.87,1.15%,-392.578086,333
6,hot,ridge_pred,857.37,1.10%,"1,119.12",1.44%,-629.649203,333
5,normal,hgb_pred,460.44,0.85%,609.60,1.12%,-280.418151,7851
4,normal,rf_pred,634.36,1.17%,849.74,1.56%,-404.737775,7851
3,normal,ridge_pred,809.16,1.49%,"1,057.05",1.94%,73.477184,7851


HGB produced the lowest MAE and RMSE across cold, normal, and hot temperature conditions.

Cold conditions were the most difficult temperature segment for all three models. HGB maintained a substantially lower error than Random Forest and Ridge, but its negative mean error indicates an average underprediction of approximately 438 MW. All models also underpredicted hot-period demand, although the bias was smaller than during cold conditions.

All the models' tendencies for larger errors and underprediction bias during unusual temperatures indicate that cold- and hot-weather forecasts still carry greater operational risk than forecasts under normal conditions.

# Conclusion

HistGradientBoosting produced the lowest overall forecast errors and performed best across normal, hot, and cold temperature conditions. It also provided the strongest results during normal and high-demand periods, making it the best general-purpose model in the final workflow.

Ridge performed best during P99+ extreme-demand hours and showed the smallest underprediction bias at the highest demand levels. This reveals an important operational tradeoff: HGB is the most accurate model across most conditions, while Ridge is more reliable at the extreme demand tail.

Forecast errors increased during unusual temperatures, and all models tended to underpredict demand during the most extreme conditions. These results support using HGB as the final forecasting model while separately monitoring extreme-demand performance and underprediction risk.

This workflow is designed primarily for short-term forecasting because lag and rolling features depend on recently observed demand. Forecasting farther into the future would require a multi-step strategy, such as recursively using earlier predictions to update these features. Recursive forecasts would also need to account for error accumulation as the forecast duration increases.

Operational use would additionally require forecasted rather than observed weather inputs. Future work could evaluate recursive forecasting, incorporate weather forecasts, improve geographic weighting, and add prediction intervals to quantify forecast uncertainty.

Overall, the project demonstrates an end-to-end forecasting workflow that combines EIA demand data, statewide weather conditions, time-aware validation, model tuning, and risk-based evaluation. Future work could incorporate weather forecasts, additional geographic weighting, and prediction intervals to better support real-time grid planning.

## Dashboard Data Preparation

Final holdout predictions and model evaluation summaries were reshaped into dashboard-ready tables and persisted in DuckDB for use in Tableau. This separates the modeling workflow from the presentation layer while preserving reproducible, queryable outputs.

In [56]:
forecast_tableau = model_results.copy()
pred_cols = ['ridge_pred', 'rf_pred', 'hgb_pred']
model_names = ['Ridge', 'RF', 'HGB']
model_name_map = dict(zip(pred_cols, model_names))

forecast_tableau = forecast_tableau.melt(
    id_vars=['period_utc', 'y_true', 'temp_mean'],
    value_vars = pred_cols,
    var_name = 'model',
    value_name = 'predicted_demand'
)

forecast_tableau = forecast_tableau.rename(columns = {
    'y_true': 'actual_demand'
})
forecast_tableau['model'] = forecast_tableau['model'].map(model_name_map)
forecast_tableau['error'] = (
    forecast_tableau['predicted_demand'] - forecast_tableau['actual_demand']
)
forecast_tableau['absolute_error'] = forecast_tableau['error'].abs()
forecast_tableau['squared_error'] = forecast_tableau['error'] ** 2

forecast_tableau['demand_segment'] = pd.cut(
    forecast_tableau['actual_demand'],
    bins = [-float('inf'), hi_demand_threshold, ex_demand_threshold, float('inf')],
    labels = ['normal','high_demand','extreme_demand']
)

forecast_tableau['temp_segment'] = 'normal'
forecast_tableau.loc[
    forecast_tableau['temp_mean'] <= train_temps.quantile(.05),
    'temp_segment'
] = 'cold'
forecast_tableau.loc[
    forecast_tableau['temp_mean'] >= train_temps.quantile(.95),
    'temp_segment'
] = 'hot'

forecast_tableau = forecast_tableau[
    [
        'period_utc',
        'model',
        'actual_demand',
        'predicted_demand',
        'error',
        'absolute_error',
        'squared_error',
        'demand_segment',
        'temp_segment'
    ]
]

forecast_tableau.head()

,period_utc,model,actual_demand,predicted_demand,error,absolute_error,squared_error,demand_segment,temp_segment
0,2025-01-01 06:00:00,Ridge,43875.0,42478.492173,-1396.507827,1396.507827,1.950234e+06,normal,normal
1,2025-01-01 07:00:00,Ridge,43161.0,41822.040152,-1338.959848,1338.959848,1.792813e+06,normal,normal
2,2025-01-01 08:00:00,Ridge,43085.0,41630.366596,-1454.633404,1454.633404,2.115958e+06,normal,normal
3,2025-01-01 09:00:00,Ridge,42870.0,41926.084290,-943.915710,943.915710,8.909769e+05,normal,normal
4,2025-01-01 10:00:00,Ridge,42709.0,42151.393537,-557.606463,557.606463,3.109250e+05,normal,normal


In [57]:
overall_rows = []

for col in pred_cols:
    error = model_results[col] - model_results['y_true']

    overall_rows.append({
        'model': model_name_map[col],
        'evaluation_type': 'overall',
        'segment': 'all',
        'hours': len(model_results),
        'mae': error.abs().mean(),
        'rmse': (error**2).mean()**.5,
        'mean_error': error.mean()
    })

eval_results_overall = pd.DataFrame(overall_rows)
eval_results_overall

,model,evaluation_type,segment,hours,mae,rmse,mean_error
0,Ridge,overall,all,8737,857.101523,1122.202116,-1.032101
1,RF,overall,all,8737,680.676802,933.916166,-443.492078
2,HGB,overall,all,8737,487.321788,648.240187,-288.131461


In [58]:
eval_results_demand_tableau = eval_results_demand.copy()

eval_results_demand_tableau['evaluation_type'] = 'demand'
eval_results_demand_tableau['model'] = eval_results_demand_tableau['model'].map(model_name_map)

eval_results_demand_tableau = eval_results_demand_tableau[
    ['model', 'evaluation_type', 'segment', 'hours', 'mae', 'rmse', 'mean_error']
]

eval_results_demand_tableau

,model,evaluation_type,segment,hours,mae,rmse,mean_error
6,Ridge,demand,extreme_demand,183,612.034538,824.219261,-305.010155
8,HGB,demand,extreme_demand,183,820.291141,1041.870460,-488.951089
7,RF,demand,extreme_demand,183,839.997759,1064.423811,-623.350438
5,HGB,demand,high_demand,970,682.371575,843.739880,-438.588856
4,RF,demand,high_demand,970,981.594376,1263.339740,-790.516722
3,Ridge,demand,high_demand,970,1012.947684,1257.982947,-242.131200
2,HGB,demand,normal,7584,454.340289,605.686391,-264.042106
1,RF,demand,normal,7584,638.344816,879.390813,-394.767397
0,Ridge,demand,normal,7584,843.082072,1109.953785,37.139591


In [59]:
eval_results_temp_tableau = eval_results_temp.copy()

eval_results_temp_tableau['evaluation_type'] = 'temp'
eval_results_temp_tableau['model'] = eval_results_temp_tableau['model'].map(model_name_map)

eval_results_temp_tableau = eval_results_temp_tableau[
    ['model', 'evaluation_type', 'segment', 'hours', 'mae', 'rmse', 'mean_error']
]

eval_results_temp_tableau

,model,evaluation_type,segment,hours,mae,rmse,mean_error
2,HGB,temp,cold,553,769.193171,972.527851,-438.244952
1,RF,temp,cold,553,1319.956320,1745.815200,-1024.349936
0,Ridge,temp,cold,553,1537.618500,1810.861383,-680.314026
8,HGB,temp,hot,333,653.034524,832.596172,-220.697326
7,RF,temp,hot,333,711.014968,893.872874,-392.578086
6,Ridge,temp,hot,333,857.371189,1119.122389,-629.649203
5,HGB,temp,normal,7851,460.438942,609.602104,-280.418151
4,RF,temp,normal,7851,634.361151,849.738944,-404.737775
3,Ridge,temp,normal,7851,809.156588,1057.050713,73.477184


In [3]:
model_eval_tableau = pd.concat(
    [
        eval_results_overall,
        eval_results_temp_tableau,
        eval_results_demand_tableau
    ],
    ignore_index=True
)
model_eval_tableau = model_eval_tableau.sort_values(
    ['evaluation_type', 'segment', 'mae'],
    ascending=[True, True, True]
).reset_index(drop=True)

model_eval_tableau

NameError: name 'eval_results_overall' is not defined

In [6]:
for df in [demand_weather_model, forecast_results]:
    df["period_utc_tableau"] = (
        pd.to_datetime(df["period_utc"], utc=True)
        .dt.tz_localize(None)
    )

NameError: name 'demand_weather_model' is not defined

In [2]:
con = duckdb.connect("ercot_weather.duckdb")

con.register('model_eval_df', model_eval_tableau)

con.execute('''
    CREATE OR REPLACE TABLE model_evaluation AS
    SELECT *
    FROM model_eval_df
''')
con.register('forecast_df', forecast_tableau)
con.execute('''
    CREATE OR REPLACE TABLE forecast_results AS
    SELECT *
    FROM forecast_df
''')

con.execute('SHOW TABLES').df()



NameError: name 'model_eval_tableau' is not defined

In [5]:
con = duckdb.connect("ercot_weather.duckdb")
con.execute("""
    COPY demand_weather_model
    TO 'demand_weather_model.csv'
    (HEADER, DELIMITER ',')
""")

con.execute("""
    COPY forecast_results
    TO 'forecast_results.csv'
    (HEADER, DELIMITER ',')
""")

con.execute("""
    COPY model_evaluation
    TO 'model_evaluation.csv'
    (HEADER, DELIMITER ',')
""")

con.close()

In [ ]:
con = duckdb.connect("ercot_weather.duckdb")

demand_weather_model = con. execute('''
''')